# Shortlister training (Role A)

QLoRA fine-tuning of the misconception shortlister as a **bi-encoder**: the query from
`build_query()` and every Eedi misconception name are embedded by the same model, and
misconceptions are ranked by cosine similarity. Misconceptions never seen in training can
still be ranked, because they are scored by their text.

**Kaggle setup**
1. Accelerator: **GPU T4 x1** (or P100). Internet: **on**.
2. *Add Input* -> the competition **eedi-mining-misconceptions-in-mathematics**.
3. Our code, either way:
   - *Add Input* -> a private dataset named **matheasy-code** holding the repo (at least `matheasy/`), or
   - *Add-ons -> Secrets* -> `GITHUB_TOKEN` with read access to Zenith053/MathEasy.

`SMOKE = True` runs the week-1 check: 1.5B model, 100 training pairs, a few steps.
Set `SMOKE = False` for the real week-2 runs.

In [ ]:
!pip install -q -U peft bitsandbytes lark rank_bm25

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/tmp/MathEasy")  # not /kaggle/working: Eedi-derived files must not end up in the saved output
if not WORK.exists():
    code_dir = next(INPUT.rglob("prepare_eedi.py"), None)  # matheasy-code dataset, wherever it is mounted
    if code_dir:
        shutil.copytree(code_dir.parents[2], WORK)  # .../matheasy/shortlister/prepare_eedi.py -> repo root
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        subprocess.run(["git", "clone", "--depth", "1",
                        f"https://{token}@github.com/Zenith053/MathEasy.git", str(WORK)], check=True)

# The scripts read Eedi from data/eedi/raw: point it at the competition input.
eedi = next(INPUT.rglob("misconception_mapping.csv")).parent
raw = WORK / "data" / "eedi" / "raw"
raw.parent.mkdir(parents=True, exist_ok=True)
if not raw.exists():
    raw.symlink_to(eedi)
print("code:", WORK, " eedi:", eedi)

os.chdir(WORK)
sys.path.insert(0, str(WORK))
!python -m matheasy.shortlister.prepare_eedi
!python -m matheasy.shortlister.query

In [ ]:
SMOKE = True

MODEL = "Qwen/Qwen2.5-1.5B-Instruct" if SMOKE else "Qwen/Qwen2.5-3B-Instruct"
N_TRAIN = 100 if SMOKE else None     # None = all training pairs
EPOCHS = 1 if SMOKE else 3
BATCH = 8                            # in-batch negatives: bigger is better if memory allows
LR = 1e-4
TEMP = 0.05
MAX_Q, MAX_M = 384, 64               # token limits for query and misconception name
LORA_R = 16
SEED = 20261016
OUT = Path("/kaggle/working/adapter-smoke" if SMOKE else "/kaggle/working/adapter")

INSTRUCT = "Instruct: Given a student's wrong step, retrieve the misconception that explains it\nQuery: "

In [ ]:
import pandas as pd
from matheasy.shortlister.shortlist import map_at_k, recall_at_k

queries = pd.read_csv("data/eedi/queries.csv")
names = pd.read_csv("data/eedi/raw/misconception_mapping.csv")
names["MisconceptionName"] = names.MisconceptionName.str.strip()

train = queries[queries.Split == "train"].sample(frac=1, random_state=SEED)
if N_TRAIN:
    train = train.head(N_TRAIN)
val = queries[queries.Split == "val"]
print(len(train), "train pairs,", len(val), "val pairs,", len(names), "candidate misconceptions")
name_of = dict(zip(names.MisconceptionId, names.MisconceptionName))

In [ ]:
import torch, torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

torch.manual_seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "right"
model = AutoModel.from_pretrained(
    MODEL,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.float16,
                                           bnb_4bit_use_double_quant=True),
    dtype=torch.float16, device_map={"": 0})
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                        gradient_checkpointing_kwargs={"use_reentrant": False})
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()


def embed(texts, max_len):
    """Last-token pooling, L2-normalised."""
    batch = tok([t + tok.eos_token for t in texts], max_length=max_len, truncation=True,
                padding=True, return_tensors="pt").to(model.device)
    hidden = model(**batch).last_hidden_state
    last = batch["attention_mask"].sum(1) - 1
    return F.normalize(hidden[torch.arange(len(texts)), last].float(), dim=-1)


@torch.no_grad()
def evaluate(df, k=25):
    model.eval()
    m = torch.cat([embed(names.MisconceptionName[i:i + 64].tolist(), MAX_M)
                   for i in range(0, len(names), 64)])
    q = torch.cat([embed([INSTRUCT + t for t in df.Query[i:i + 16]], MAX_Q)
                   for i in range(0, len(df), 16)])
    top = (q @ m.T).topk(k, dim=1).indices.cpu()
    ranked = [[names.MisconceptionId.iloc[j] for j in row] for row in top.tolist()]
    true = df.MisconceptionId.tolist()
    model.train()
    return map_at_k(ranked, true, k), recall_at_k(ranked, true, 10)

In [ ]:
import time
print("before training: val MAP@25 = %.3f  Recall@10 = %.3f" % evaluate(val))

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
scaler = torch.amp.GradScaler()
model.train()
start = time.time()
for epoch in range(EPOCHS):
    shuffled = train.sample(frac=1, random_state=SEED + epoch)
    for i in range(0, len(shuffled) - BATCH + 1, BATCH):
        b = shuffled.iloc[i:i + BATCH]
        with torch.autocast("cuda", dtype=torch.float16):
            q = embed([INSTRUCT + t for t in b.Query], MAX_Q)
            p = embed([name_of[m] for m in b.MisconceptionId], MAX_M)
        logits = (q @ p.T) / TEMP
        # Two rows with the same misconception are not negatives of each other.
        ids = torch.tensor(b.MisconceptionId.values, device=logits.device)
        same = (ids[:, None] == ids[None, :]) & ~torch.eye(len(b), dtype=torch.bool, device=logits.device)
        loss = F.cross_entropy(logits.masked_fill(same, -1e4), torch.arange(len(b), device=logits.device))
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        if (i // BATCH) % 5 == 0:
            print(f"epoch {epoch} step {i // BATCH:4d}  loss {loss.item():.3f}  {time.time() - start:.0f}s")
    print("epoch %d: val MAP@25 = %.3f  Recall@10 = %.3f" % ((epoch,) + evaluate(val)))

In [ ]:
model.save_pretrained(OUT)
tok.save_pretrained(OUT)
print("saved adapter ->", OUT, sorted(p.name for p in OUT.iterdir()))
print("BM25 placeholder on val for comparison: MAP@25 = 0.070, Recall@10 = 0.138")

**Smoke test passes when:** the loss goes down, both evaluations print, and the adapter folder
is saved. With only 100 pairs the MAP@25 is not meaningful yet; the week-2 runs are.